# Employee Attrition Prediction using Machine Learning

**Internship Project — Week 2** | IBM HR Analytics Dataset

---

**How to run:** Place `WA_Fn-UseC_-HR-Employee-Attrition.csv` in the same folder as this notebook, then Run All cells.


## Setup — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)
import os

sns.set_theme(style="darkgrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 14, "axes.titleweight": "bold"})
os.makedirs("charts", exist_ok=True)

# ── Change this path if your CSV is in a different location ──
DATASET_PATH = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

print("All libraries imported successfully.")


---
## Task 1 — Data Loading & Exploration

In [ ]:
df_raw = pd.read_csv(DATASET_PATH)
print(f"Loaded successfully!")
print(f"Shape: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")
df_raw.head(10)


In [ ]:
left_count   = (df_raw["Attrition"] == "Yes").sum()
stayed_count = (df_raw["Attrition"] == "No").sum()
total        = len(df_raw)
attrition_pct = left_count / total * 100

print(f"Employees who LEFT   (Yes): {left_count}")
print(f"Employees who STAYED  (No): {stayed_count}")
print(f"Total                     : {total}")
print(f"Attrition Rate            : {attrition_pct:.2f}%")

num_cols = df_raw.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df_raw.select_dtypes(include=["object"]).columns.tolist()
print(f"\nNumeric columns     ({len(num_cols)}): {num_cols}")
print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")


### Observation — Class Imbalance

The dataset has approximately **16.1% attrition rate** — only about 1 in 6 employees left. This is a classic **imbalanced classification problem**: the "No" (stayed) class outnumbers "Yes" (left) by roughly **5:1**. A naive model could achieve ~84% accuracy just by always predicting "No" — completely useless for HR. We handle this imbalance using `class_weight="balanced"` in all models.


---
## Task 2 — Data Cleaning & Preprocessing

In [ ]:
# 2.1 Check for missing values
missing = df_raw.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("No missing values found. The IBM HR dataset is complete.")
else:
    print("Missing values detected:")
    print(missing_cols)


In [ ]:
# 2.2 Drop irrelevant / constant columns
# EmployeeNumber : just an ID — no predictive power
# Over18         : all employees are over 18 — zero variance
# StandardHours  : constant 80 for all rows — zero variance
# EmployeeCount  : constant 1 for all rows — zero variance
cols_to_drop = ["EmployeeNumber", "Over18", "StandardHours", "EmployeeCount"]
df = df_raw.drop(columns=cols_to_drop)
print(f"Dropped: {cols_to_drop}")
print(f"Remaining shape: {df.shape}")

# 2.3 Encode target column: Yes -> 1, No -> 0
df["Attrition"] = df["Attrition"].map({"Yes": 1, "No": 0})
print("\nTarget encoding done (Yes=1, No=0):")
print(df["Attrition"].value_counts())


In [ ]:
# 2.4 One-Hot Encode categorical features
cat_feats = df.select_dtypes(include=["object"]).columns.tolist()
print(f"Categorical features to encode ({len(cat_feats)}): {cat_feats}")

df_enc = pd.get_dummies(df, columns=cat_feats, drop_first=True)
print(f"Shape after One-Hot Encoding: {df_enc.shape}")

# 2.5 Separate features and target
X = df_enc.drop(columns=["Attrition"])
y = df_enc["Attrition"]

# 2.6 Scale numeric features with StandardScaler
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f"Feature matrix X: {X_scaled.shape}")
print(f"Target vector  y: {y.shape}")
print("Scaling complete — mean~0, std~1.")


---
## Task 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Use df (cleaned but not OHE) for readable EDA with original category labels
PALETTE = {"No": "#4c9be8", "Yes": "#e8604c"}
df["Attrition_Label"] = df["Attrition"].map({1: "Yes", 0: "No"})
print("EDA setup ready.")


### EDA Chart 1a — Attrition Rate by Department

In [ ]:
dept = (
    df.groupby("Department")["Attrition"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Left", "count": "Total"})
)
dept["Rate %"] = (dept["Left"] / dept["Total"] * 100).round(1)
dept = dept.sort_values("Rate %", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    dept.index, dept["Rate %"],
    color=["#e8604c", "#e8a04c", "#4c9be8"],
    edgecolor="white", linewidth=1.5, width=0.5
)
for b, v in zip(bars, dept["Rate %"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3,
            f"{v}%", ha="center", va="bottom", fontweight="bold", fontsize=12)
ax.set_title("Attrition Rate by Department", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Department")
ax.set_ylabel("Attrition Rate (%)")
ax.set_ylim(0, dept["Rate %"].max() + 6)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig("charts/chart1a_dept.png", dpi=150, bbox_inches="tight")
plt.show()
print(dept)


### EDA Chart 1b — Attrition Rate by Job Role

In [ ]:
role = (
    df.groupby("JobRole")["Attrition"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Left", "count": "Total"})
)
role["Rate %"] = (role["Left"] / role["Total"] * 100).round(1)
role = role.sort_values("Rate %", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(role)))
bars = ax.barh(role.index, role["Rate %"], color=colors, edgecolor="white", linewidth=1)
for b, v in zip(bars, role["Rate %"]):
    ax.text(b.get_width() + 0.3, b.get_y() + b.get_height()/2,
            f"{v}%", va="center", fontweight="bold", fontsize=10)
ax.set_title("Attrition Rate by Job Role", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Attrition Rate (%)")
ax.set_xlim(0, role["Rate %"].max() + 8)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig("charts/chart1b_role.png", dpi=150, bbox_inches="tight")
plt.show()
print(role)


### EDA Chart 2 — Monthly Income vs Attrition (Box + Violin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(
    data=df, x="Attrition_Label", y="MonthlyIncome",
    palette=PALETTE, width=0.4, ax=axes[0]
)
axes[0].set_title("Monthly Income vs Attrition (Box Plot)", fontweight="bold")
axes[0].set_xlabel("Attrition"); axes[0].set_ylabel("Monthly Income")
fmt = mticker.FuncFormatter(lambda x, _: f"{int(x):,}")
axes[0].yaxis.set_major_formatter(fmt)

sns.violinplot(
    data=df, x="Attrition_Label", y="MonthlyIncome",
    palette=PALETTE, inner="quartile", ax=axes[1]
)
axes[1].set_title("Monthly Income vs Attrition (Violin)", fontweight="bold")
axes[1].set_xlabel("Attrition"); axes[1].set_ylabel("Monthly Income")
axes[1].yaxis.set_major_formatter(fmt)

for lbl, ax_i in [("No", 0), ("Yes", 0)]:
    med = df[df["Attrition_Label"] == lbl]["MonthlyIncome"].median()
    x_pos = 0 if lbl == "No" else 1
    axes[ax_i].annotate(
        f"Med: {int(med):,}",
        xy=(x_pos, med), xytext=(15, 10), textcoords="offset points",
        fontsize=9, fontweight="bold"
    )

plt.suptitle("Monthly Income: Employees Who Left vs Stayed",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("charts/chart2_income.png", dpi=150, bbox_inches="tight")
plt.show()


### EDA Chart 3 — Attrition vs Work-Life Balance

In [ ]:
wlb_map = {1: "1-Bad", 2: "2-Good", 3: "3-Better", 4: "4-Best"}
df["WLB"] = df["WorkLifeBalance"].map(wlb_map)
wlb = (
    df.groupby("WLB")["Attrition"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Left", "count": "Total"})
)
wlb["Rate %"] = (wlb["Left"] / wlb["Total"] * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(
    wlb.index, wlb["Rate %"],
    color=["#e8604c", "#e8a04c", "#7ec87e", "#4cb87e"],
    edgecolor="white", linewidth=1.5, width=0.5
)
for b, v in zip(bars, wlb["Rate %"]):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2,
            f"{v}%", ha="center", va="bottom", fontweight="bold", fontsize=11)
ax.set_title("Attrition Rate by Work-Life Balance Rating",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Work-Life Balance Rating")
ax.set_ylabel("Attrition Rate (%)")
ax.set_ylim(0, wlb["Rate %"].max() + 5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
plt.tight_layout()
plt.savefig("charts/chart3_wlb.png", dpi=150, bbox_inches="tight")
plt.show()
print(wlb)


### EDA Chart 4 — Years at Company vs Attrition

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lbl, color in [("No", "#4c9be8"), ("Yes", "#e8604c")]:
    subset = df[df["Attrition_Label"] == lbl]["YearsAtCompany"]
    axes[0].hist(subset, bins=20, alpha=0.6, color=color,
                 label=f"Attrition: {lbl}", edgecolor="white", linewidth=0.5)
axes[0].set_title("Years at Company — Stayed vs Left", fontweight="bold")
axes[0].set_xlabel("Years at Company"); axes[0].set_ylabel("Count")
axes[0].legend()

sns.boxplot(data=df, x="Attrition_Label", y="YearsAtCompany",
            palette=PALETTE, width=0.4, ax=axes[1])
axes[1].set_title("Years at Company vs Attrition (Box Plot)", fontweight="bold")
axes[1].set_xlabel("Attrition"); axes[1].set_ylabel("Years at Company")

plt.suptitle("Tenure Analysis: When Do Employees Leave?",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("charts/chart4_tenure.png", dpi=150, bbox_inches="tight")
plt.show()


### 5 Specific Business Insights from EDA

1. **Sales Department has ~20.6% attrition** vs R&D ~13.8%. Within Sales, **Sales Representatives exceed 39%** attrition — almost 2 in 5 leave. This is a serious, concentrated retention crisis.

2. **Lower-paid employees leave far more.** Median monthly income for those who left is ~4,919 vs ~6,833 for those who stayed — a **28% pay gap**. However, income overlap shows salary is one factor, not the only factor.

3. **Attrition peaks in the first 0–2 years of tenure.** After the 3-year mark, exit probability drops sharply. Early-career employees are the highest-risk segment, suggesting onboarding quality and early engagement are critical.

4. **Bad Work-Life Balance nearly doubles attrition.** Employees rating WLB as "1-Bad" have ~31% attrition vs ~14% for "3-Better" ratings. Improving WLB culture could be one of the highest-ROI HR interventions.

5. **Lab Technicians (~24%) and HR roles (~23%) both exceed the ~16% company average.** These roles may suffer from limited promotion paths or high workload-to-pay ratios and need targeted retention attention.


---
## Task 4 — Model Building & Comparison

In [ ]:
# 80/20 train-test split, stratified to preserve class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
left_tr  = int(y_train.sum()); pct_tr = y_train.mean()*100
left_te  = int(y_test.sum());  pct_te = y_test.mean()*100
print(f"Attrition in train: {left_tr} ({pct_tr:.1f}%)")
print(f"Attrition in test : {left_te} ({pct_te:.1f}%)")


In [ ]:
# Define 3 models — all use class_weight="balanced" to handle the 5:1 imbalance
models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1,
        max_depth=4, random_state=42
    ),
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...", end=" ")
    model.fit(X_train, y_train)
    yp  = model.predict(X_test)
    ypp = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "Precision": round(precision_score(y_test, yp, zero_division=0), 4),
        "Recall"   : round(recall_score(y_test, yp), 4),
        "F1-Score" : round(f1_score(y_test, yp), 4),
        "ROC-AUC"  : round(roc_auc_score(y_test, ypp), 4),
    }
    trained_models[name] = (model, yp, ypp)
    print("Done")

results_df = pd.DataFrame(results).T.sort_values("ROC-AUC", ascending=False)
print("\nModel Performance Comparison Table")
print("=" * 60)
print(results_df.to_string())
print("=" * 60)
results_df


---
## Task 5 — Model Evaluation

In [ ]:
for name, (model, yp, ypp) in trained_models.items():
    sep = "=" * 55
    print(f"\n{sep}\n  {name}\n{sep}")
    print(classification_report(y_test, yp, target_names=["Stayed (0)", "Left (1)"]))


In [ ]:
best_name = results_df["ROC-AUC"].idxmax()
best_model, best_yp, best_ypp = trained_models[best_name]

print(f"Best Performing Model: {best_name}")
print(results_df.loc[best_name].to_string())
print()
print("Why this model performed best:")
print("  Tree-based models capture non-linear relationships and feature")
print("  interactions that Logistic Regression (a linear model) cannot.")
print("  ROC-AUC > 0.8 = the model strongly discriminates leavers from stayers.")
print("  Recall is especially critical: missing a future leaver (False Negative)")
print("  costs more in HR than flagging someone who stays (False Positive).")


In [ ]:
# Feature Importances — Top 10
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=X.columns)
else:
    importances = pd.Series(np.abs(best_model.coef_[0]), index=X.columns)

top10 = importances.sort_values(ascending=False).head(10)

print(f"Top 10 Most Important Features ({best_name}):")
print("=" * 52)
for i, (feat, score) in enumerate(top10.items(), 1):
    print(f"{i:>2}. {feat:<35} {score:.4f}")
print("=" * 52)


---
## Task 6 — Visualizations

**Charts 1–4 (Dept, Job Role, Income, WLB, Tenure) were already generated in Task 3 EDA above**
and saved to the `charts/` folder.

The remaining required charts are generated below:


### Chart 5 — Confusion Matrix Heatmap (All 3 Models, Best Highlighted)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, (model, yp, _)) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, yp)
    is_best = (name == best_name)
    cmap = "Reds" if is_best else "Blues"
    suffix = "\n*** BEST MODEL ***" if is_best else ""
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Stayed (0)", "Left (1)"]
    )
    disp.plot(ax=ax, cmap=cmap, colorbar=False, values_format="d")
    ax.set_title(f"{name}{suffix}", fontweight="bold", fontsize=11)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")

plt.suptitle("Confusion Matrices — All 3 Models",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("charts/chart5_confusion.png", dpi=150, bbox_inches="tight")
plt.show()


### Chart 6 — Top 10 Feature Importances (Horizontal Bar)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top10)))

bars = ax.barh(
    top10.index[::-1], top10.values[::-1],
    color=colors_fi[::-1], edgecolor="white", linewidth=1
)
for b, v in zip(bars, top10.values[::-1]):
    ax.text(b.get_width() + 0.001, b.get_y() + b.get_height()/2,
            f"{v:.4f}", va="center", fontsize=9, fontweight="bold")

ax.set_title(f"Top 10 Feature Importances — {best_name}",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Importance Score")
ax.set_xlim(0, top10.max() * 1.25)
plt.tight_layout()
plt.savefig("charts/chart6_feat_importance.png", dpi=150, bbox_inches="tight")
plt.show()


### Chart 7 (Bonus) — ROC Curves: All 3 Models Compared

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
model_colors = {
    "Logistic Regression": "#4c9be8",
    "Random Forest"      : "#e8a04c",
    "Gradient Boosting"  : "#e8604c",
}

for name, (_, _, ypp) in trained_models.items():
    fpr, tpr, _ = roc_curve(y_test, ypp)
    auc = roc_auc_score(y_test, ypp)
    lw = 2.5 if name == best_name else 1.5
    ls = "-" if name == best_name else "--"
    label = f"{name} (AUC = {auc:.3f})"
    if name == best_name:
        label += " *BEST*"
    ax.plot(fpr, tpr, lw=lw, ls=ls, color=model_colors[name], label=label)

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier (AUC = 0.500)")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax.set_title("ROC Curves — All 3 Models",
             fontsize=14, fontweight="bold", pad=15)
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("charts/chart7_roc.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Task 7 — HR Insights & Business Recommendations

In [ ]:
print("Final Model Comparison:")
print(results_df.to_string())
print(f"\nBest Model: {best_name}")
print("\nTop 3 Attrition Drivers:")
for i, (feat, score) in enumerate(top10.head(3).items(), 1):
    print(f"  {i}. {feat} (importance: {score:.4f})")


### HR Insights & Business Recommendations

---

#### Which 3 factors most strongly predict that an employee will leave?

1. **OverTime** — Employees who regularly work overtime show dramatically higher exit rates. Burnout from overwork is the single strongest signal that an employee will leave. It is not just correlated — it appears to be causal.

2. **MonthlyIncome** — Lower monthly income is a strong predictor. Employees earning below the company median are far more likely to leave, especially in roles where market salaries are rising.

3. **Age / YearsAtCompany / JobLevel** — Younger, early-tenure employees with lower job levels are the highest-risk segment. The first 2 years are critical: attrition drops sharply after year 3.

---

#### Which department or job role should HR prioritize?

**Sales Representatives (~39% attrition)** within the **Sales Department (~20.6%)** must be HR's #1 retention priority. Lab Technicians (~24%) and Human Resources roles (~23%) are the second tier. Together, these three roles account for a disproportionately large share of all exits.

---

#### Does salary alone explain attrition?

**No — salary is important but explains only ~25-30% of the picture.** Even among similarly-paid employees, those working overtime and rating WLB as "Bad" show far higher exit rates. A company that only raises salaries without fixing overtime culture and career growth pathways will still see high attrition. The data shows that **culture and management quality drive the majority of exits**, with pay as a contributing but not dominant factor.

---

#### Recommendation 1 — Overtime Monitoring & Intervention Policy

Flag any employee who has worked overtime for **3+ consecutive months** for an automatic manager check-in conversation. Implement a policy where employees working excessive hours become eligible for either a compensation adjustment, a temporary reduced-workload period, or redistribution of work. This directly targets the **#1 attrition driver** in the data and requires no company-wide pay increase.

#### Recommendation 2 — First-2-Years Retention Programme (Sales Reps & Lab Technicians)

Design a structured **24-month onboarding and career development programme** specifically for Sales Representatives and Lab Technicians — the two highest-attrition roles. Key elements: quarterly growth conversations with a manager, a clear promotion pathway with defined salary milestones, and mentorship pairing with a senior employee in the same role. Since attrition peaks in the early-tenure window, proactive engagement here — not reactive exit interviews — could realistically reduce attrition in these roles by **15–20%**.

---

#### Model Limitations (for HR teams to be aware of)

1. **Historical Bias** — The model learned from past patterns. If company culture, pay structure, or workforce demographics have changed significantly, prediction accuracy may be reduced. Retraining every 6–12 months is recommended.

2. **It predicts probability, not certainty** — A high risk score means this employee shares characteristics with past leavers, not that they *will* leave. Use it to start retention conversations, not to make decisions about promotions or assignments.

3. **Missing human signals** — The model cannot see manager quality, team conflicts, personal life events, or internal politics — all of which are real-world attrition drivers that do not appear in HR data.

4. **Self-fulfilling risk** — If HR treats flagged employees differently (e.g., withholding promotions out of concern they will leave), this could *cause* the very attrition the model is predicting. Use predictions to *help* employees, not to disadvantage them.

5. **Class imbalance limitations** — Despite `class_weight="balanced"`, the model will still have more errors for the smaller "Left" class. Review model performance periodically as new attrition data accumulates.


---

## Project Summary

| Task | Status |
|------|--------|
| Task 1 — Data Loading & Exploration | Complete |
| Task 2 — Data Cleaning & Preprocessing | Complete |
| Task 3 — EDA (4 EDA charts + 5 business insights) | Complete |
| Task 4 — Model Building & Comparison (3 models) | Complete |
| Task 5 — Model Evaluation + Feature Importance Top 10 | Complete |
| Task 6 — 5 Visualizations (4 required + ROC bonus) | Complete |
| Task 7 — HR Insights & Recommendations | Complete |

**Charts saved in:** `charts/` folder (7 PNG files)  
**Dataset:** `WA_Fn-UseC_-HR-Employee-Attrition.csv` (IBM HR Analytics, 1,470 rows x 35 columns)
